import

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon,MultiPolygon,LineString,LinearRing,MultiPoint,MultiLineString#,Point,box
from shapely.plotting import plot_polygon, plot_line
from shapely.affinity import scale
from shapely import affinity
import numpy as np
from shapely.ops import unary_union
from Data_Process.utils import *
from Data_Process.SwissD_DataProcess import *
import os
import imageio
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

load data

In [ ]:
data = pd.read_csv("Data/geometries.csv")
area_subtypes_file_path = "Data_Process/room_type.csv"
area_subtypes_df = pd.read_csv(area_subtypes_file_path)

# Extract the room_types list
room_types = area_subtypes_df.loc[area_subtypes_df['room'] == 1, 'entity_subtype'].tolist()
apartment_list = data['apartment_id'].unique()
floor_list = data['floor_id'].unique()

house_index = (area_subtypes_df['house'] != 0)
floor_index = (area_subtypes_df['publicspace'] != 0)
house_room_types = area_subtypes_df.loc[house_index, 'entity_subtype'].tolist()
floor_room_types = area_subtypes_df.loc[floor_index, 'entity_subtype'].tolist()
house_type_index = area_subtypes_df.loc[house_index, 'house'].tolist()

attribute init

In [ ]:
img_res = 128 # img res: 128*128*4

# 0: None 1: Living 2: Bath 3: CLoset 4: Bed 5: Kitchen 6: Dining 7: Balcony 8: Corridor 9: end 10: start
# Not used: 9: FOYER and stair 10: Garden and wintergarten 11: outdoor 12: facilities (duct, pipe, etc.)

T_dim = 10 # 10 one-hot
L_dim = 7 # 7 bits, max 128
S_dim = 6 # 6 bits, max 64
R_num = 20
divide = 32

max_room = 14
W_num = max_room
max_boundary_length = 20 # in real-world meters
num_plan = 45000

# +2: start and end token
parsed_type = np.zeros((num_plan,max_room+2,T_dim))
parsed_loc = np.zeros((num_plan,max_room+2,2,L_dim))
parsed_ada = np.zeros((num_plan,max_room+2,max_room))
parsed_area = np.zeros((num_plan,max_room+2,S_dim))
parsed_room = np.zeros((num_plan,max_room+2,R_num,2,L_dim))
parsed_window = np.zeros((num_plan,W_num+2,2,L_dim))

# assign start vec value
parsed_type[:,0,-1] = 1
parsed_loc[:,0,:,:] = 1
parsed_ada[:,0,:] = 1
parsed_area[:,0,:] = 1
parsed_room[:,0,:,:,:] = 1
parsed_window[:,0,:,:] = 1

create folder path

In [ ]:
os.makedirs('Data/img/all', exist_ok=True) # all img
os.makedirs('Data/img/site', exist_ok=True) # site img
os.makedirs('Data/img/composed', exist_ok=True) # composed img
os.makedirs('Data/img/composed_all', exist_ok=True) # composed all img
os.makedirs('Data/sqe', exist_ok=True) # sequence as np
attribute_list = ['T','S','L','A','R','W'] # 6 attributes
for j in attribute_list:
    for i in range(max_room+2):
        os.makedirs('Data/img/%s/%d' % (j,i), exist_ok=True)

color list

In [ ]:
# 0: None 1: Living 2: Bath 3: CLoset 4: Bed 5: Kitchen 6: Dining 7: Balcony 8: Corridor 9: end 10: start
# Not used: 9: FOYER and stair 10: Garden and wintergarten 11: outdoor 12: facilities (duct, pipe, etc.)

color_list = [[0, 0, 0, 0], # 0: None
 [127, 127, 255, 255], # 1: Living
 [0, 255, 255, 255], # 2: Bath
 [127, 255, 0, 255], # 3: CLoset
 [255, 0, 255, 255], # 4: Bed
 [255, 127, 127, 255], # 5: Kitchen
 [255, 0, 127, 255], # 6: Dining
 [127, 255, 127, 255], # 7: Balcony
 [255, 255, 0, 255], # 8: Corridor
 [127, 127, 127, 127], # 9: end
 [255, 255, 255, 255], # 10: start
] 

Parse Floorplan

In [ ]:
SwissD = Swiss_Dewelling_Dual_Modality(max_boundary_length,color_list,house_room_types,house_type_index,
                                       img_res,max_room,T_dim,L_dim,S_dim,R_num,max_room,divide)

indx = 0
skip = [21720,31339,33694] # invalid geometry

for j,k in enumerate(apartment_list):
    if j in skip: continue
    else: 
        tmp = data.loc[data['apartment_id']==k]

        if j%100 == 0:
            print(j)

        if tmp['site_id'].unique().shape[0] == 1:

            if tmp['floor_id'].unique().shape[0] == 0:
                contest_poly = Polygon([[0,0],[0,1],[0,0]])
                floor_id = 0
            else:
                floor_id = tmp['floor_id'].unique()[0]
                contest = data.loc[data['floor_id'] == floor_id]
                contest_poly = parse_contest(contest.index,contest['geometry'])
            # try:
            sucess = SwissD.transfer_polygons(tmp,indx,contest_poly)

            if sucess == 1:

                parsed_type[indx,1:] = SwissD.T_vec
                parsed_loc[indx,1:] = SwissD.L_vec
                parsed_ada[indx,1:] = SwissD.A_vec
                parsed_area[indx,1:] = SwissD.S_vec
                parsed_room[indx,1:] = SwissD.R_vec
                parsed_window[indx,1:] = SwissD.W_vec
                indx += 1
                SwissD.reset()

            # except:
            #     SwissD.reset()

        else:
            for m,n in enumerate(tmp['site_id'].unique()):

                tmps = data.loc[data['site_id']==n]

                if tmps['floor_id'].unique().shape[0] == 0:
                    contest_poly = Polygon([[0,0],[0,1],[0,0]])
                    floor_id = 0
                else:
                    floor_id = tmps['floor_id'].unique()[0]
                    contest = data.loc[data['floor_id'] == floor_id]
                    contest_poly = parse_contest(contest.index,contest['geometry'])
                # try:
                sucess = SwissD.transfer_polygons(tmp,indx,contest_poly)

                if sucess == 1:
                    
                    parsed_type[indx,1:] = SwissD.T_vec
                    parsed_loc[indx,1:] = SwissD.L_vec
                    parsed_ada[indx,1:] = SwissD.A_vec
                    parsed_area[indx,1:] = SwissD.S_vec
                    parsed_room[indx,1:] = SwissD.R_vec
                    parsed_window[indx,1:] = SwissD.W_vec
                    indx += 1
                    SwissD.reset()
                
                # except:
                #     SwissD.reset()

In [ ]:
corner_distri = np.zeros((20))
for s in range(indx):
    for r in range(14):
        for c in range(20):
            if np.array_equal(parsed_room[s,r+1,c],np.zeros((2,L_dim))) and c != 0: 
                corner_distri[c] += 1
                break


In [ ]:
plt.figure(figsize=(20, 10))
plt.bar(np.arange(20),corner_distri)
plt.show()

In [ ]:
np.savez_compressed('Data/sqe/swissD_Dual_Modal.npz', T = parsed_type[:indx].astype(np.uint8), L = parsed_loc[:indx].astype(np.uint8), A = parsed_ada[:indx].astype(np.uint8),
          S = parsed_area[:indx].astype(np.uint8), R = parsed_room[:indx].astype(np.uint8), W = parsed_window[:indx].astype(np.uint8))

In [ ]:
np.save('Data/sqe/new_R_10.npy',parsed_room[:indx,:,:10].astype(np.uint8))

save imgs batch to imgs

In [ ]:
import numpy as np
import os
import cv2
from tensorflow.keras.preprocessing import image

os.environ["OPENCV_LOG_LEVEL"]="FATAL"
vec_data = np.load('Data/sqe/swissD_Dual_Modal.npz')
os.makedirs('Data/img/Batch',exist_ok=True)
file_index = vec_data.files

vec_T = vec_data[file_index[0]].astype(np.int32)

num_data = vec_T.shape[0]
attribute_list = ['T','S','L','A','R','W']

for i in range(42643,num_data*3):

    init = np.zeros((6,14,128,128,4))
    room_num = np.where(vec_T[i % num_data][:,8]==1)[0][0]-1

    for j in range(room_num):
        for n,m in enumerate(attribute_list):

            if n < 5:
                init[n,j] = cv2.imread('Data/img/%s/%d/%d.png' % (m,j+1,i % num_data),cv2.IMREAD_UNCHANGED)
            else:
                try:
                    init[n,j] = image.load_img('Data/img/W/%d/%d.png' % (j+1,i % num_data), color_mode='rgba', target_size=(128, 128))
                except:
                    continue 
    
    cv2.imwrite('Data/img/Batch/%d.png'% (i),init.reshape((6*14*128,128,4)))
    if i % 100 == 0: print(i)

save imgs batch to numpy idx (low efficient)

In [ ]:
import numpy as np
import os
from tensorflow.keras.preprocessing import image
os.environ["OPENCV_LOG_LEVEL"]="FATAL"
vec_data = np.load('Data/sqe/swissD_Dual_Modal.npz')
file_index = vec_data.files

vec_T = vec_data[file_index[0]].astype(np.int32)

num_data = vec_T.shape[0]

# [id,num_room,window_indexs]
# shape[1+1+14]
indexs = np.zeros((num_data,16))
for i in range(num_data):
    room_num = np.where(vec_T[i][:,8]==1)[0][0]-1
    indexs[i,0] = i
    indexs[i,1] = room_num
    for j in range(room_num):

        try:
            path = 'Data/img/W/%d/%d.png' % (j+1,i)
            img = image.load_img(path, color_mode='rgba', target_size=(128, 128))
            indexs[i,2+j] = 1
        except:
            continue 
        
    if i % 100 == 0: print(i)

indexs_3 = np.concatenate((indexs,indexs,indexs),axis=0)
np.save('Data/img_batch_3.npy',indexs_3.astype(np.int32))

In [ ]:
import numpy as np
from itertools import combinations

def euclidean_distance(set1, set2):
    """
    Compute the Euclidean distance between two sets.
    """
    return np.linalg.norm(np.array(set1) - np.array(set2))

def compute_distance_matrix(sets):
    """
    Compute the pairwise distance matrix for all sets.
    """
    N = len(sets)
    D = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            D[i, j] = euclidean_distance(sets[i], sets[j])
            D[j, i] = D[i, j]  # Distance matrix is symmetric
    return D

def most_diverse_4_sets(sets):
    """
    Find the 4 most diverse sets from a list of sets.
    """
    N = len(sets)
    if N < 4:
        raise ValueError("At least 4 sets are required.")

    # Compute the distance matrix
    D = compute_distance_matrix(sets)

    # Initialize the selected sets
    selected = []
    remaining = list(range(N))

    # Step 1: Select the two sets with the maximum distance
    max_dist = -1
    for i, j in combinations(remaining, 2):
        if D[i, j] > max_dist:
            max_dist = D[i, j]
            selected = [i, j]
    remaining = [x for x in remaining if x not in selected]

    # Step 2: Iteratively add the set that maximizes the total distance
    for _ in range(2):
        max_total_dist = -1
        best_candidate = -1
        for candidate in remaining:
            total_dist = sum(D[candidate, s] for s in selected)
            if total_dist > max_total_dist:
                max_total_dist = total_dist
                best_candidate = candidate
        selected.append(best_candidate)
        remaining.remove(best_candidate)

    # Return the selected sets
    return [sets[i] for i in selected]

sets = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12],
    [1, 1, 1],
    [2, 2, 2],
    [3, 3, 3],
]

# Find the 4 most diverse sets
diverse_sets = most_diverse_4_sets(sets)
print("Most diverse 4 sets:")
for s in diverse_sets:
    print(s)